# Reproducing Das et al. (2019): normal vs abnormal medulloblastoma tissue from texture features

Goal: rerun the protocol of *Automated classification of childhood brain tumours based on texture feature*
(Songklanakarin J. Sci. Technol. 41(5)) on the public IEEE DataPort images, then repeat it with a split
that keeps each tumour patient on one side. Nothing here is trained with a neural network.

**What follows the paper:** 10x images only; red channel; CLAHE; five texture families (GLCM, GRLN, histogram,
Tamura, LBP) evaluated alone and combined; six classifiers (tree, LD, QD, LR, SVM, KNN k=10); features standardised;
5-fold cross-validation over images.

**What cannot be matched, stated up front**
- The paper used 80 normal + 80 abnormal images at 10x. The DataPort set has 50 normal and about 152 abnormal at 10x
  (202 in total). Normal is subsampled against abnormal here, over several random draws.
- The authors used MATLAB File Exchange feature code (172/171/174 features are quoted in three places). This notebook
  re-implements the feature definitions in Python and gets 135 features. Gray levels, LBP variant and the Tamura
  directionality step are my choices where the paper is silent.
- The paper does not say how many patients the 160 images came from or whether any split respected patients.
- Normal images carry no patient label (file names are just `N (k)`), so the patient-held-out run can only hold out
  *tumour* patients. Its normal-class numbers are still image-level and can be optimistic.


In [1]:
import os, re, glob, warnings, random, time
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2
from skimage.feature import graycomatrix, local_binary_pattern
from scipy.stats import skew, kurtosis
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

SEED = 42
random.seed(SEED); np.random.seed(SEED)

DATA_DIR = r"C:\Users\Shafique\Documents\uni\healthcare project\final_database"
OUTPUT_DIR = "./outputs/das_repro"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MAGS = ["10x", "100x"]          # the manifest lists both; the reproduction uses REPRO_MAG only
REPRO_MAG = "10x"
IMG_EXT = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
IMG_SIZE_WH = (512, 384)         # size quoted in the paper for 10x images
N_DRAWS = 10                     # random normal/abnormal subsamples for the balanced runs

# Tumour slides and their subtype (only used to tell abnormal from unknown; Das's task is binary).
SLIDE_LABELS = {
    "411": "Desmoplastic_Nodular", "203": "Desmoplastic_Nodular", "141": "Desmoplastic_Nodular",
    "321": "Classic", "279": "Classic", "277": "Classic", "1766A": "Classic", "1766B": "Classic", "188": "Classic",
    "17126": "Largecell", "395": "Largecell",
}
EXCLUDED_IDS = {"1609"}          # no diagnosis in the spreadsheet
# Counts from database.xlsx / the DataPort page (Desmoplastic + Nodular merged): per magnification.
SHEET = {("Classic", "10x"): 59, ("Desmoplastic_Nodular", "10x"): 65, ("Largecell", "10x"): 28, ("Normal", "10x"): 50,
         ("Classic", "100x"): 82, ("Desmoplastic_Nodular", "100x"): 22, ("Largecell", "100x"): 24, ("Normal", "100x"): 25}
DATAPORT_TOTALS = {"10x": 202, "100x": 153}


## 1. Manifest (every image file is listed; nothing is dropped silently)

The slide id is whatever precedes the `(n)` in the file name, with fallbacks for other naming styles. A file whose id
is not recognised stays in the table with role `unknown`, and its id and an example file name are printed below.


In [2]:
def parse_id(fname):
    stem = os.path.splitext(fname)[0].strip()
    m = re.match(r"^(.*?)\s*\(\s*\d+\s*\)\s*$", stem) or re.match(r"^(.*?)[\s_\-]*\d+\s*$", stem)
    sid = m.group(1).strip() if (m and m.group(1).strip()) else stem
    return sid.replace(" ", "").upper()

rows, non_images = [], {}
for mag in MAGS:
    d = os.path.join(DATA_DIR, mag)
    if not os.path.isdir(d):
        print("[warn] folder not found:", d); continue
    for fp in sorted(glob.glob(os.path.join(d, "*"))):
        if os.path.isdir(fp):
            continue
        if not fp.lower().endswith(IMG_EXT):
            non_images.setdefault(mag, []).append(os.path.basename(fp)); continue
        sid = parse_id(os.path.basename(fp))
        if sid == "N":                     role, label = "normal", "Normal"
        elif sid in SLIDE_LABELS:          role, label = "abnormal", SLIDE_LABELS[sid]
        elif sid in EXCLUDED_IDS:          role, label = "excluded", None
        else:                              role, label = "unknown", None
        rows.append(dict(filepath=fp, magnification=mag, slide_id=sid, role=role, label=label,
                         fname=os.path.basename(fp)))
man = pd.DataFrame(rows)

print("Images per role and magnification:")
print(man.groupby(["role", "magnification"]).size().unstack(fill_value=0))
print("\nTotal image files:", man.groupby("magnification").size().to_dict(),
      "| DataPort page says:", DATAPORT_TOTALS)
print("\nPer slide id:")
print(man.groupby(["slide_id", "magnification"]).size().unstack(fill_value=0))

chk = man[man.label.notna()].groupby(["label", "magnification"]).size()
cmp_ = pd.DataFrame({"found": chk, "sheet": pd.Series(SHEET)}).fillna(0).astype(int)
cmp_["diff"] = cmp_["found"] - cmp_["sheet"]
print("\nFound vs spreadsheet counts (diff != 0 needs a look):")
print(cmp_)

other = man[man.role.isin(["unknown", "excluded"])]
if len(other):
    print("\nFiles NOT used (excluded or unrecognised id):")
    print(other.groupby(["role", "slide_id", "magnification"]).agg(n=("fname", "size"), example=("fname", "first")))
if non_images:
    print("\nNon-image files ignored:", {k: v[:5] for k, v in non_images.items()})


Images per role and magnification:
magnification  100x  10x
role                    
abnormal        128  154
excluded          4    0
normal           25   50
unknown           1    0

Total image files: {'100x': 158, '10x': 204} | DataPort page says: {'10x': 202, '100x': 153}

Per slide id:
magnification  100x  10x
slide_id                
141               2   23
1609              4    0
17126             8    2
1766A             6    6
1766B             4    4
188              12    5
203               7   17
277               5    6
279              29    8
279(              1    0
321              26   30
395              16   28
411              13   25
N                25   50

Found vs spreadsheet counts (diff != 0 needs a look):
                           found  sheet  diff
Classic              100x     82     82     0
                     10x      59     59     0
Desmoplastic_Nodular 100x     22     22     0
                     10x      65     65     0
Largecell            

## 2. Features

Red channel, resized to 512x384, CLAHE (clip 2.0, 8x8 tiles), then:

| family | what | count |
|---|---|---|
| GLCM | 22 Haralick-type statistics (Uppuluri's list), distance 1, four angles, 8 gray levels | 88 |
| GRLN | gray-level run-length: SRE, LRE, GLN, RLN, RP, LGRE, HGRE, four directions, 16 gray levels | 28 |
| HIST | first-order histogram: mean, variance, skewness, kurtosis, energy, entropy | 6 |
| TAMURA | coarseness, contrast, directionality (directionality is a simplified single-peak version) | 3 |
| LBP | uniform LBP (P=8, R=1) histogram, 10 bins | 10 |


In [3]:
EPS = 1e-12

def haralick22(P):
    P = P / (P.sum() + EPS)
    G = P.shape[0]
    idx = np.arange(1, G + 1)
    i = idx[:, None]; j = idx[None, :]
    px, py = P.sum(1), P.sum(0)
    mux, muy = (idx * px).sum(), (idx * py).sum()
    sx = np.sqrt(((idx - mux) ** 2 * px).sum()); sy = np.sqrt(((idx - muy) ** 2 * py).sum())
    p_sum = np.zeros(2 * G - 1); p_diff = np.zeros(G)
    for a in range(G):
        for b in range(G):
            p_sum[a + b] += P[a, b]; p_diff[abs(a - b)] += P[a, b]
    k_sum = np.arange(2, 2 * G + 1); k_diff = np.arange(0, G)
    sum_avg = (k_sum * p_sum).sum()
    diff_mean = (k_diff * p_diff).sum()
    HX = -(px * np.log(px + EPS)).sum(); HY = -(py * np.log(py + EPS)).sum()
    HXY = -(P * np.log(P + EPS)).sum()
    pxpy = px[:, None] * py[None, :]
    HXY1 = -(P * np.log(pxpy + EPS)).sum(); HXY2 = -(pxpy * np.log(pxpy + EPS)).sum()
    d = np.abs(i - j)
    return np.array([
        (i * j * P).sum(),                                           # autocorrelation
        ((i - j) ** 2 * P).sum(),                                    # contrast
        ((i * j * P).sum() - mux * muy) / (sx * sy + EPS),           # correlation
        ((i + j - mux - muy) ** 4 * P).sum(),                        # cluster prominence
        ((i + j - mux - muy) ** 3 * P).sum(),                        # cluster shade
        (d * P).sum(),                                               # dissimilarity
        (P ** 2).sum(),                                              # energy
        HXY,                                                         # entropy
        (P / (1 + (i - j) ** 2)).sum(),                              # homogeneity
        P.max(),                                                     # maximum probability
        ((i - mux) ** 2 * P).sum(),                                  # sum of squares
        ((j - muy) ** 2 * P).sum(),                                  # variance (other marginal)
        sum_avg,                                                     # sum average
        ((k_sum - sum_avg) ** 2 * p_sum).sum(),                      # sum variance
        -(p_sum * np.log(p_sum + EPS)).sum(),                        # sum entropy
        ((k_diff - diff_mean) ** 2 * p_diff).sum(),                  # difference variance
        -(p_diff * np.log(p_diff + EPS)).sum(),                      # difference entropy
        (HXY - HXY1) / max(HX, HY, EPS),                             # information measure of correlation 1
        np.sqrt(max(0.0, 1 - np.exp(-2 * (HXY2 - HXY)))),            # information measure of correlation 2
        (P / (1 + d)).sum(),                                         # inverse difference
        (P / (1 + d / G)).sum(),                                     # inverse difference normalised
        (P / (1 + ((i - j) / G) ** 2)).sum(),                        # inverse difference moment normalised
    ])

def _rl_matrix(q, G, direction):
    H, W = q.shape
    if direction == 0:     lines = [q[r, :] for r in range(H)]
    elif direction == 90:  lines = [q[:, c] for c in range(W)]
    elif direction == 135: lines = [np.diagonal(q, o) for o in range(-(H - 1), W)]
    else:                  lines = [np.diagonal(q[:, ::-1], o) for o in range(-(H - 1), W)]
    maxrun = max(H, W)
    flat = []
    for ln in lines:
        if ln.size == 0: continue
        b = np.flatnonzero(np.diff(ln)) + 1
        starts = np.concatenate(([0], b)); ends = np.concatenate((b, [ln.size]))
        flat.append(ln[starts].astype(np.int64) * maxrun + (ends - starts - 1))
    R = np.bincount(np.concatenate(flat), minlength=G * maxrun).reshape(G, maxrun).astype(np.float64)
    return R

def run_length_features(R, n_pixels):
    G, L = R.shape
    gi = np.arange(1, G + 1)[:, None]; rj = np.arange(1, L + 1)[None, :]
    N = R.sum() + EPS
    return [(R / rj ** 2).sum() / N, (R * rj ** 2).sum() / N,          # SRE, LRE
            (R.sum(1) ** 2).sum() / N, (R.sum(0) ** 2).sum() / N,      # GLN, RLN
            N / n_pixels,                                              # RP
            (R / gi ** 2).sum() / N, (R * gi ** 2).sum() / N]          # LGRE, HGRE

def hist_features(g8):
    v = g8.ravel().astype(np.float64)
    p = np.bincount(g8.ravel(), minlength=256) / v.size
    return [v.mean(), v.var(), skew(v), kurtosis(v, fisher=False), (p ** 2).sum(), -(p[p > 0] * np.log2(p[p > 0])).sum()]

def tamura_features(g8):
    g = g8.astype(np.float32)
    E = []
    for k in range(1, 6):
        w = 2 ** k; h = w // 2
        A = cv2.blur(g, (w, w))
        Eh = np.abs(np.roll(A, -h, axis=1) - np.roll(A, h, axis=1))
        Ev = np.abs(np.roll(A, -h, axis=0) - np.roll(A, h, axis=0))
        E.append(np.maximum(Eh, Ev))
    coarse = (2.0 ** (np.stack(E).argmax(0) + 1)).mean()
    sd = g.std()
    contrast = 0.0 if sd < 1e-6 else sd / (((g - g.mean()) ** 4).mean() / sd ** 4) ** 0.25
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    mag = (np.abs(gx) + np.abs(gy)) / 2
    theta = np.arctan2(gy, gx) % np.pi
    sel = mag > np.percentile(mag, 50)                   # strongest half of the gradients
    hist, edges = np.histogram(theta[sel], bins=16, range=(0, np.pi))
    hist = hist / (hist.sum() + EPS)
    centres = (edges[:-1] + edges[1:]) / 2
    dphi = np.abs(centres - centres[hist.argmax()]); dphi = np.minimum(dphi, np.pi - dphi)
    direction = 1 - (dphi ** 2 * hist).sum() / (np.pi / 2) ** 2
    return [coarse, contrast, direction]

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
GROUPS = ["GLCM", "GRLN", "HIST", "TAMURA", "LBP"]
GROUP_LEN = {"GLCM": 88, "GRLN": 28, "HIST": 6, "TAMURA": 3, "LBP": 10}

def extract(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None: raise FileNotFoundError(path)
    red = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)[:, :, 0]
    red = _clahe.apply(cv2.resize(red, IMG_SIZE_WH, interpolation=cv2.INTER_AREA))
    q8 = (red // 32).astype(np.uint8)
    glcm = graycomatrix(q8, [1], [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4], levels=8, symmetric=True, normed=True)
    f_glcm = np.concatenate([haralick22(glcm[:, :, 0, a]) for a in range(4)])
    q16 = (red // 16).astype(np.uint8)
    f_grln = np.concatenate([run_length_features(_rl_matrix(q16, 16, d), red.size) for d in (0, 45, 90, 135)])
    f_hist = np.array(hist_features(red)); f_tam = np.array(tamura_features(red))
    lbp = local_binary_pattern(red, 8, 1, "uniform")
    f_lbp = np.histogram(lbp, bins=10, range=(0, 10), density=True)[0]
    return np.concatenate([f_glcm, f_grln, f_hist, f_tam, f_lbp]).astype(np.float32)

_o = np.cumsum([0] + [GROUP_LEN[g] for g in GROUPS])
GROUP_COLS = {g: np.arange(_o[k], _o[k + 1]) for k, g in enumerate(GROUPS)}
GROUP_COLS["ALL5"] = np.arange(_o[-1])


In [4]:
use = man[(man.magnification == REPRO_MAG) & (man.role.isin(["normal", "abnormal"]))].reset_index(drop=True)
cache = os.path.join(OUTPUT_DIR, f"das_features_{REPRO_MAG}.npz")
paths = use["filepath"].tolist()
X = None
if os.path.exists(cache):
    z = np.load(cache, allow_pickle=False)
    lk = {p: i for i, p in enumerate(z["paths"].tolist())}
    if all(p in lk for p in paths):
        X = z["feats"][[lk[p] for p in paths]]
if X is None:
    t0 = time.time()
    X = np.stack([extract(p) for p in paths])
    np.savez(cache, paths=np.array(paths), feats=X)
    print(f"extracted {X.shape} in {time.time() - t0:.0f}s")
y = (use.role == "abnormal").astype(int).values            # 0 normal, 1 abnormal
# 1766A and 1766B are treated as one patient (conservative; the spreadsheet lists them as two samples).
groups = use.slide_id.replace({"1766A": "1766", "1766B": "1766"}).values
print("features:", X.shape, "| normal:", int((y == 0).sum()), "| abnormal:", int((y == 1).sum()),
      "| NaN:", int(np.isnan(X).sum()))


extracted (204, 135) in 33s
features: (204, 135) | normal: 50 | abnormal: 154 | NaN: 0


## 3. Das protocol: stratified 5-fold CV over images, balanced normal/abnormal

Each draw takes all normal images and an equally large random sample of abnormal images, then runs 5-fold CV.
Reported: mean accuracy over draws. The last block compares the combined-features row with the paper's Table 1.


In [5]:
CLASSIFIERS = ["Tree", "LD", "QD", "LR", "SVM", "KNN"]
def _make_qd():
    # More features than samples per class makes the class covariances singular. Newer scikit-learn raises
    # unless shrinkage is set; older versions (e.g. 1.2.x) only have reg_param. Strength 0.5 is my choice.
    try:
        return QuadraticDiscriminantAnalysis(priors=[0.5, 0.5], solver="eigen", shrinkage=0.5)
    except TypeError:
        return QuadraticDiscriminantAnalysis(priors=[0.5, 0.5], reg_param=0.5)

def make_clf(name):
    clf = {"Tree": DecisionTreeClassifier(max_leaf_nodes=3, class_weight="balanced", random_state=SEED),   # max 2 splits
           "LD":   LinearDiscriminantAnalysis(priors=[0.5, 0.5]),
           "QD":   _make_qd(),
           "LR":   LogisticRegression(max_iter=5000, class_weight="balanced"),
           "SVM":  SVC(kernel="linear", class_weight="balanced"),
           "KNN":  KNeighborsClassifier(n_neighbors=10)}[name]
    return make_pipeline(StandardScaler(), clf)

def das_cv(X, y, n_draws=N_DRAWS):
    norm_idx = np.flatnonzero(y == 0); abn_idx = np.flatnonzero(y == 1)
    n = min(len(norm_idx), len(abn_idx))
    acc = {(c, g): [] for c in CLASSIFIERS for g in GROUP_COLS}
    for draw in range(n_draws):
        rng = np.random.default_rng(SEED + draw)
        sel = np.concatenate([rng.choice(norm_idx, n, replace=False), rng.choice(abn_idx, n, replace=False)])
        Xs, ys = X[sel], y[sel]
        skf = StratifiedKFold(5, shuffle=True, random_state=SEED + draw)
        for g, cols in GROUP_COLS.items():
            for c in CLASSIFIERS:
                pred = np.empty_like(ys)
                for tr, te in skf.split(Xs, ys):
                    pred[te] = make_clf(c).fit(Xs[tr][:, cols], ys[tr]).predict(Xs[te][:, cols])
                acc[(c, g)].append((pred == ys).mean())
    return acc

accA = das_cv(X, y)
tabA = pd.DataFrame({g: {c: np.mean(accA[(c, g)]) * 100 for c in CLASSIFIERS} for g in GROUP_COLS})
sdA = pd.DataFrame({g: {c: np.std(accA[(c, g)]) * 100 for c in CLASSIFIERS} for g in GROUP_COLS})
print(f"A. Image-level 5-fold CV, balanced {min((y==0).sum(), (y==1).sum())}+{min((y==0).sum(), (y==1).sum())} images, "
      f"{N_DRAWS} draws. Accuracy %, mean:")
print(tabA.round(1))
print("\nstd over draws:"); print(sdA.round(1))

das_all5 = pd.Series({"Tree": 100, "LD": 97.2, "QD": 99.1, "LR": 95.4, "SVM": 100, "KNN": 100})
print("\nAll 5 families combined: paper (Table 1) vs this run")
print(pd.DataFrame({"paper": das_all5, "this run": tabA["ALL5"].round(1)}))


A. Image-level 5-fold CV, balanced 50+50 images, 10 draws. Accuracy %, mean:
      GLCM  GRLN  HIST  TAMURA   LBP  ALL5
Tree  98.1  97.6  93.0    95.6  97.1  98.2
LD    93.2  97.9  90.8    92.2  98.9  97.5
QD    98.4  95.5  87.2    92.4  98.5  98.5
LR    98.5  98.1  90.5    92.5  98.5  98.5
SVM   98.3  98.4  92.6    92.4  98.6  98.8
KNN   96.5  96.1  93.8    95.0  98.3  96.8

std over draws:
      GLCM  GRLN  HIST  TAMURA  LBP  ALL5
Tree   1.0   1.4   1.6     1.5  2.0   1.2
LD     4.0   0.9   2.7     1.4  0.5   1.3
QD     0.9   1.9   1.7     1.1  0.8   0.8
LR     0.8   0.9   2.7     1.8  0.8   0.8
SVM    0.9   0.5   2.2     1.5  0.7   0.6
KNN    1.0   1.1   1.9     1.4  0.9   1.0

All 5 families combined: paper (Table 1) vs this run
      paper  this run
Tree  100.0      98.2
LD     97.2      97.5
QD     99.1      98.5
LR     95.4      98.5
SVM   100.0      98.8
KNN   100.0      96.8


## 4. Same features, tumour patients held out

Leave-one-tumour-patient-out. In each fold the test set is every image of one tumour slide plus about 1/10 of the
normal images; training uses everything else. Normal images have no patient label, so the normal side is split at
random between folds and stays image-level. The informative numbers are therefore the **tumour recall per held-out
patient**: for a patient the model has never seen, how often is the tissue called abnormal.


In [6]:
def patient_heldout(X, y, groups, cols, clf_name):
    abn_groups = sorted(set(groups[y == 1]))
    rng = np.random.default_rng(SEED)
    norm_idx = rng.permutation(np.flatnonzero(y == 0))
    chunks = np.array_split(norm_idx, len(abn_groups))
    per_patient, norm_hits, norm_n = {}, 0, 0
    for g, chunk in zip(abn_groups, chunks):
        te_abn = np.flatnonzero((y == 1) & (groups == g))
        te = np.concatenate([te_abn, chunk]); tr = np.setdiff1d(np.arange(len(y)), te)
        pred = make_clf(clf_name).fit(X[tr][:, cols], y[tr]).predict(X[te][:, cols])
        per_patient[g] = float((pred[:len(te_abn)] == 1).mean())
        norm_hits += int((pred[len(te_abn):] == 0).sum()); norm_n += len(chunk)
    return per_patient, norm_hits / norm_n

resB = {(c, g): patient_heldout(X, y, groups, cols, c) for g, cols in GROUP_COLS.items() for c in CLASSIFIERS}
tumB = pd.DataFrame({g: {c: np.mean(list(resB[(c, g)][0].values())) * 100 for c in CLASSIFIERS} for g in GROUP_COLS})
nrmB = pd.DataFrame({g: {c: resB[(c, g)][1] * 100 for c in CLASSIFIERS} for g in GROUP_COLS})
balB = (tumB + nrmB) / 2
print("B. Tumour recall % (mean over held-out tumour patients):"); print(tumB.round(1))
print("\nNormal recall % (image-level, optimistic):"); print(nrmB.round(1))
print("\nBalanced accuracy % = mean of the two:"); print(balB.round(1))

print("\nPer-patient tumour recall, all five families combined:")
print(pd.DataFrame({c: resB[(c, "ALL5")][0] for c in CLASSIFIERS}).round(2))

print("\nAll 5 combined: image-level CV (A) vs tumour-patient held out (B, balanced accuracy)")
print(pd.DataFrame({"A image-level": tabA["ALL5"], "B patient-held-out": balB["ALL5"]}).round(1))


B. Tumour recall % (mean over held-out tumour patients):
      GLCM  GRLN  HIST  TAMURA   LBP  ALL5
Tree  96.0  93.3  69.3    93.0  94.3  95.0
LD    96.0  96.0  76.0    80.0  96.0  96.0
QD    96.0  82.3  70.0    85.0  96.0  96.0
LR    94.0  94.0  72.0    80.0  96.0  95.0
SVM   95.0  94.0  70.0    80.0  96.0  96.0
KNN   93.0  92.0  80.0    85.0  96.0  94.0

Normal recall % (image-level, optimistic):
       GLCM   GRLN  HIST  TAMURA    LBP   ALL5
Tree   96.0   96.0  96.0    96.0   94.0   98.0
LD     96.0  100.0  86.0    92.0  100.0  100.0
QD     84.0   98.0  84.0    92.0  100.0   50.0
LR    100.0  100.0  90.0    90.0  100.0  100.0
SVM   100.0  100.0  92.0    92.0  100.0  100.0
KNN   100.0  100.0  90.0    96.0  100.0  100.0

Balanced accuracy % = mean of the two:
      GLCM  GRLN  HIST  TAMURA   LBP  ALL5
Tree  96.0  94.7  82.7    94.5  94.2  96.5
LD    96.0  98.0  81.0    86.0  98.0  98.0
QD    90.0  90.2  77.0    88.5  98.0  73.0
LR    97.0  97.0  81.0    85.0  98.0  97.5
SVM   97.5  97

## 5. How to read this

- Table A tests whether the paper's numbers reproduce under the paper's protocol. Expect small differences from
  the different image count (50 vs 80 normal) and the re-implemented features.
- Table B is the same features with tumour patients held out. A large drop from A to B means image-level splits were
  rewarding slide recognition. A small drop means the normal/abnormal difference carries across patients,
  which is plausible for an easier task than subtype classification.
- Normal recall in B is still optimistic (no normal patient labels). One caveat on top: the spreadsheet lists 3 normal samples,
  and Das et al. say normal regions were cut from tumour slides where possible, so the normal images may share slides
  with tumour images. Slide-specific colour then cannot separate the two classes, which makes this task harder to
  shortcut than the subtype task.
- Features are fit inside each fold (scaler included). Nothing here is tuned on the held-out data.
